In [1]:
import os
import PIL.Image
import pytesseract
import pdf2image
from typing import Literal, Optional
import re
import pandas as pd
import cv2
import numpy as np

## Convert pdf to images

In [2]:
def pdf_to_img(pdf_file, dpi: int=300):
    return pdf2image.convert_from_path(pdf_file, dpi=300)

def save_images(imgs, output_base="./output", file_format: Literal["PNG", "JPG"] = "PNG"):
    os.makedirs(output_base, exist_ok=True)
    for i, img in enumerate(imgs):
        img.save(os.path.join(output_base, f"page_{i}.{file_format}"), file_format)

In [3]:
pdfFile = "input/MinnaNoNihongoSolutionsPartB.pdf"
imgs = pdf_to_img(pdfFile)

## Perform OCR with Tesseract

In [4]:
def ocr_core(img, language: str = "jpn", custom_config: Optional[str] = None):
    if custom_config is None:
        custom_config = rf'--oem 3 --psm 6 -l {language}'
    text = pytesseract.image_to_string(img, lang=language)
    return text

def output_text(imgs, custom_config: Optional[str] = None):
    if custom_config is None:
        custom_config = r'--oem 3 --psm 6 -l jpn'
    for i, img in enumerate(imgs):
        text = pytesseract.image_to_string(img, config=custom_config)
        print(f"Page {i+1} Extracted Text:\n", text)

In [ ]:
text = ocr_core(imgs[1])
text2 = ocr_core(imgs[2])
print(text)

In [ ]:
# save_images(imgs)
# output_text(imgs)

In [ ]:
print(text)

In [ ]:
print(text2)

## Extract Structured Data
Since the document has a clear structure (第X課, 練習B, X., 1) 2) ...), we can use regular expressions to extract information.

**Regex patterns**:
- **Chapters**: `第(\d+)課`
- **Section B**: `練習B`
- **Exercises**: `(\d+)\.` (number followed by a dot)
- **Sentences**: `(\d+)\)` (number followed by a close round bracket)

In [5]:
def extract_structured_text(text):
    structured_data = {}
    current_chapter = None
    current_exercise = None

    lines = text.split("\n")
    for line in lines:
        print(line)
        a = re.match("^練習 B$", line)
        print(a)
        # Detect chapter
        chapter_match = re.search(r'第(\d+)課', line)
        if chapter_match:
            current_chapter = int(chapter_match.group(1))
            structured_data[current_chapter] = {}

        # Do not continue until a chapter has been identified
        if current_chapter is None:
            continue

        # Detect "練習B"
        if "練習 B" in line:
            print("Found Renshuu B")
            structured_data[current_chapter]["練習 B"] = {}

        # Detect exercises (number followed by a dot)
        exercise_match = re.match(r'^(\d+)\.', line.strip())
        if exercise_match:
            current_exercise = int(exercise_match.group(1))
            structured_data[current_chapter]["練習 B"][current_exercise] = []

        # Detect sentences (number followed by `)`)
        sentence_matches = re.findall(r'(\d+)\)', line)
        if sentence_matches and current_exercise:
            sentences = re.split(r'\d+\)', line)[1:]  # Remove numbers
            sentences = [s.strip() for s in sentences if s.strip()]
            structured_data[current_chapter]["練習 B"][current_exercise].extend(sentences)

    return structured_data

In [ ]:
struc = extract_structured_text(text)

## Store in Excel for Easy Access
Once we have structured data, we save it in an Excel file for reloading later.

In [6]:
def save_to_excel(data, filename="extracted_text.xlsx"):
    rows = []

    for chapter, sections in data.items():
        for section, exercises in sections.items():
            for exercise, sentences in exercises.items():
                for sentence_num, sentence in enumerate(sentences, 1):
                    rows.append([chapter, section, exercise, sentence_num, sentence])

    df = pd.DataFrame(rows, columns=["Chapter", "Section", "Exercise", "Sentence Number", "Sentence"])
    df.to_excel(filename, index=False)

### Reload Data for Easy Access

In [7]:
def get_sentence(chapter, exercise, sentence_number, filename="extracted_text.xlsx"):
    df = pd.read_excel(filename)
    result = df[
        (df["Chapter"] == chapter) &
        (df["Exercise"] == exercise) &
        (df["Sentence Number"] == sentence_number)
    ]
    return result["Sentence"].values[0] if not result.empty else None

In [ ]:
# Example usage
sentence = get_sentence(3, 5, 2)  # Get sentence 2 of exercise 5 in chapter 3
print(sentence)

## Optional: Improve OCR Accuracy (WIP)
### _Currently not working_
If OCR results are messy, you can:

- Preprocess images (binarization, noise removal) before passing to Tesseract.
- Use a better OCR engine like Google Vision API for improved accuracy.

**Example of binarization:**

In [8]:
def preprocess_image(pil_img):
    # Convert PIL image to grayscale first
    pil_gray = pil_img.convert("L")  # "L" mode means grayscale

    # Convert grayscale PIL image to NumPy array
    np_gray = np.array(pil_gray)

    # Ensure it's a contiguous array
    np_gray = np.ascontiguousarray(np_gray)

    # Apply binary thresholding (Otsu's method)
    _, binary = cv2.threshold(np_gray, 150, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    return binary

def binarized_ocr(img):
    bin_img = preprocess_image(img)
    text = ocr_core(bin_img)
    return text

# Handling Furigana
Furigana (small kana characters above or beside kanji) can interfere with OCR because Tesseract treats them as regular text, leading to incorrect recognition. Since your document contains furigana, we need a way to remove or ignore them before extracting the main text.
## Possible Solutions
1. Use Image Preprocessing to Remove Small Characters

    **Steps:**
    - Convert the image to grayscale.
    - Apply adaptive thresholding to enhance contrast.
    - Use contour detection to remove small elements.
    - Run OCR on the cleaned image.
3. Post-Processing: Remove Extra Kana Characters from OCR Output

    **Steps:**
    - Extract OCR text.
    - Identify kanji with surrounding hiragana (e.g., using regex or frequency analysis).
    - Remove small hiragana that appear out of place.
3. Use a Better OCR Model or different Tesseract config

    **Steps:**
    - Try OCR with --psm 11 (Sparse text OCR mode).
    - Compare results with --psm 6 (Default block OCR mode).
    - Use a dictionary to validate words.

## Preprocessing to Remove Small Furigana

In [25]:
def remove_furigana(image_path):
    """Preprocess image to remove small furigana text using OpenCV only."""
    
    # Read image using OpenCV (no PIL)
    img = cv2.imread(image_path, cv2.IMREAD_COLOR)

    # Convert to grayscale
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Apply adaptive thresholding for better text segmentation
    thresh = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                   cv2.THRESH_BINARY_INV, 25, 15)

    # Find contours
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    # Remove small text (likely furigana)
    for cnt in contours:
        x, y, w, h = cv2.boundingRect(cnt)
        if h < 15:  # Adjust size threshold to remove small text
            cv2.rectangle(gray, (x, y), (x + w, y + h), (255, 255, 255), -1)

    return gray

def remove_furigana_v2(image_path):
    """Remove small furigana without affecting diacritics or punctuation."""

    # Read image using OpenCV
    img = cv2.imread(image_path, cv2.IMREAD_COLOR)

    # Convert to grayscale
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Apply adaptive thresholding (enhance contrast)
    thresh = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                   cv2.THRESH_BINARY_INV, 25, 15)

    # Detect contours
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    # Compute median height of all text blocks
    heights = [cv2.boundingRect(cnt)[3] for cnt in contours]
    median_height = np.median(heights)

    # Remove only small text *above* kanji, keeping punctuation & diacritics
    for cnt in contours:
        x, y, w, h = cv2.boundingRect(cnt)

        # Condition: Likely furigana if it's small and positioned above a larger block
        if h < median_height * 0.6:  # Small text (likely furigana)
            # Ensure it's positioned above another text block (to avoid diacritics)
            below_y = y + h + 5  # Check a bit below the detected text
            below_region = gray[below_y:below_y + 5, x:x + w]

            # If there's a solid region below, it's furigana → remove it
            if np.mean(below_region) < 200:  # Darker means text is present
                cv2.rectangle(gray, (x, y), (x + w, y + h), (255, 255, 255), -1)

    return gray

In [26]:
processed_img = remove_furigana_v2("output/page_1.PNG")

In [27]:
# Ensure image is in correct format for pytesseract
custom_config = r'--oem 3 --psm 6 -l jpn'
text = pytesseract.image_to_string(processed_img, config=custom_config)

In [28]:
print(text)

れんしゅっ      かいとうれい
練習B・C 解答例
第 26 課
れんしゅう
練 泊 B やま                          の
1. 1) 山へ行くんですか。 2) エレベーターに径ちらないんですか<。
つく             れむ
3) シュュッ ト さんか作ったんですか。   4) 過いんですか。
ゃし          と            きんか >  と
2. 1) きれいな写真ですね。どこで撮ったんですか。……金 関寺で撮りました。
え
2) おもしろい絵ですね。だれがかいたんですか。 ーーカリナさんがかきま し
なに                 ぼ  ど   れんしゅう
3) ずいぶんにぎやかですれね。何をやっているんですか。……全踊りの練 習 をやって
います。
にほんご  じょうず      、   、  へんきょう             ねんべんきょう
4) 日本語が上手ですね。どのく<く らい免 強 したんですか。……2年免 強 しました。
えい 、、  わ
3. 1) どうしたんですか。……財布を定れたんです。
2) どう2したんですか。……かぎかないんです。
き ぶ   る
3) どうしたんですか。……気分が悪いんです。
キネっ 3  で
4) どうしたんですか。 切符が出ないんです。  靖
ひ  こ             い\
4. 1) どう して引っ越しするんですか。 ……今のうちは狭いんです。
2) どうしてケーキを食べないんですか。……ダイエットをしているんです。
、   かい ま あ           しんかんせん わく
3) どうして会議に聞に合わなかったんですか。 ーー新幹線が遅れたんです。
はや 。 かえ           つま たんじょう ひ
4) どうして早く帰るんですか。……き ょ 2は妻の説 生 日なんです。
い            と
5. 1) いいえ、 あまり人行きません。 2 ちから加いんです。
パっこう  こ
2) いいえ、会いませんでした。タワポンさんは学校へ来なかったんです。
かい
3) すみません。 これから会滅なんです。
や
4 ) すみません。 き ょ 2はちょっと約束があるんです。
やくしょ い
6. 1) 市化所へ行きたいんですが、 地図をの